<a href="https://colab.research.google.com/github/aarushkote/Myprojects/blob/main/AML_Experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import files
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 1. Load the dataset
# Since you have the file locally, this will prompt you to upload 'iris_synthetic_data.csv'
print("Please upload the 'iris_synthetic_data.csv' file:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

Please upload the 'iris_synthetic_data.csv' file:


In [ ]:
# 2. Perform EDA (Exploratory Data Analysis)
print("\n--- Step 2: Exploratory Data Analysis ---")
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nData Types and Info:")
print(df.info())
print("\nStatistical Summary:")
print(df.describe())

In [ ]:
# 3. Display and Handle Missing Values
print("\n--- Step 3: Handling Missing Values ---")
print("Missing values before handling:\n", df.isnull().sum())

# Handling Strategy: Fill numerical with median, categorical with mode
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

print("Missing values after handling:", df.isnull().sum().sum())

In [ ]:
# 4. Display and Handle Outliers (using sepal width as an example)
print("\n--- Step 4: Handling Outliers (Sepal Width) ---")
target_col = 'sepal width'

plt.figure(figsize=(10, 4))
sns.boxplot(x=df[target_col])
plt.title(f"Outliers in {target_col} Before Treatment")
plt.show()

# IQR Technique to handle outliers
Q1 = df[target_col].quantile(0.25)
Q3 = df[target_col].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Capping outliers to the upper and lower bounds
df[target_col] = np.where(df[target_col] > upper_bound, upper_bound,
                 np.where(df[target_col] < lower_bound, lower_bound, df[target_col]))

print(f"Outliers in {target_col} have been capped using IQR.")

In [ ]:
# 5. Data Encoding (Converting 'label' to numerical)
print("\n--- Step 5: Data Encoding ---")
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])
print("Encoded 'label' categories:", dict(zip(le.classes_, le.transform(le.classes_))))

In [ ]:
# 6. Feature Scaling
print("\n--- Step 6: Feature Scaling ---")
scaler = StandardScaler()

# Separate features and target
X = df.drop('label', axis=1)
y = df['label']

print("Head BEFORE Scaling:")
print(X.head())

# Apply Scaling
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print("\nHead AFTER Scaling:")
print(X_scaled.head())

In [ ]:
#EXP-2
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import files
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, chi2, f_classif, mutual_info_classif, VarianceThreshold

# 1. Load the dataset (same as Experiment 1)
print("Please upload the 'iris_synthetic_data.csv' file:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)


# 2. Preprocess the dataset
# Impute missing values with median
df = df.fillna(df.median(numeric_only=True))

# Encode target variable 'label' to numerical values
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

# Separate features (X) and target (y)
X = df.drop(['label', 'label_encoded'], axis=1)
y = df['label_encoded']

print("\n--- Feature Selection Experiment Started ---")

# Define a function to display top features for each method
def display_top_features(method_name, scores, feature_names):
    feature_scores = pd.Series(scores, index=feature_names)
    top_features = feature_scores.sort_values(ascending=False)
    print(f"\n[Method: {method_name}]")
    print(top_features)
    return top_features

# Method 1: Variance Threshold (Filters low-variance features)
vt = VarianceThreshold()
vt.fit(X)
display_top_features("Variance Threshold", vt.variances_, X.columns)

# Method 2: Pearson Correlation (Feature correlation with target)
corr_scores = X.corrwith(y).abs()
display_top_features("Pearson Correlation (Absolute)", corr_scores.values, X.columns)

# Method 3: ANOVA F-test (Linear dependency between feature and target)
f_selector = SelectKBest(score_func=f_classif, k='all')
f_selector.fit(X, y)
display_top_features("ANOVA F-test", f_selector.scores_, X.columns)

# Method 4: Mutual Information (Information gain - captures any dependency)
mi_scores = mutual_info_classif(X, y, random_state=42)
display_top_features("Mutual Information", mi_scores, X.columns)

# Method 5: Chi-Square Test (Measures association for non-negative features)
chi2_selector = SelectKBest(score_func=chi2, k='all')
chi2_selector.fit(X, y)
display_top_features("Chi-Square Test", chi2_selector.scores_, X.columns)

# Visualizing the scores comparison
plt.figure(figsize=(12, 6))
sns.heatmap(X.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix for Feature Selection")
plt.show()


In [ ]:
#EXP-3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score

# 1. Load the dataset
print("Please upload the 'iris_synthetic_data.csv' file:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)


# 2. Preprocess the dataset
# Fill missing values
df = df.fillna(df.median(numeric_only=True))

# Encode target label
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])

# Define Features and Target
X = df.drop('label', axis=1)
y = df['label']

# Scaling is important for Logistic Regression convergence
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Split into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)


# 3. Initialize the Model (Logistic Regression)
model = LogisticRegression()

print("\n--- Step 3: Sequential Forward Selection (SFS) ---")
# We will select the top 2 features
sfs = SequentialFeatureSelector(model, n_features_to_select=2, direction='forward')
sfs.fit(X_train, y_train)

sfs_features = X.columns[sfs.get_support()].tolist()
print(f"Top Features selected by SFS: {sfs_features}")

# Train and evaluate SFS model
X_train_sfs = sfs.transform(X_train)
X_test_sfs = sfs.transform(X_test)
model.fit(X_train_sfs, y_train)
sfs_acc = accuracy_score(y_test, model.predict(X_test_sfs))

print("\n--- Step 4: Sequential Backward Elimination (SBE) ---")
# We will select the top 2 features using backward direction
sbe = SequentialFeatureSelector(model, n_features_to_select=2, direction='backward')
sbe.fit(X_train, y_train)

sbe_features = X.columns[sbe.get_support()].tolist()
print(f"Top Features selected by SBE: {sbe_features}")

# Train and evaluate SBE model
X_train_sbe = sbe.transform(X_train)
X_test_sbe = sbe.transform(X_test)
model.fit(X_train_sbe, y_train)
sbe_acc = accuracy_score(y_test, model.predict(X_test_sbe))


# 4. Compare Performance
print("\n--- Step 5: Performance Comparison ---")
comparison_df = pd.DataFrame({
    'Method': ['Forward Selection (SFS)', 'Backward Elimination (SBE)'],
    'Selected Features': [", ".join(sfs_features), ", ".join(sbe_features)],
    'Accuracy Score': [sfs_acc, sbe_acc]
})

print(comparison_df)

# Visualizing the comparison
plt.figure(figsize=(8, 5))
plt.bar(comparison_df['Method'], comparison_df['Accuracy Score'], color=['skyblue', 'salmon'])
plt.ylim(0, 1.1)
plt.ylabel('Accuracy Score')
plt.title('Comparison of SFS vs SBE Model Performance')
for i, v in enumerate(comparison_df['Accuracy Score']):
    plt.text(i, v + 0.02, f"{v:.4f}", ha='center', fontweight='bold')
plt.show()


In [ ]:
#EXP-4
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Load the dataset
print("Please upload the 'iris_synthetic_data.csv' file:")
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)


# 2. Preprocess the dataset
df = df.fillna(df.median(numeric_only=True))
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

X = df.drop(['label', 'label_encoded'], axis=1)
y = df['label_encoded']

# PCA and LDA are sensitive to scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# 3. Apply PCA (Unsupervised)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])


# 4. Apply LDA (Supervised)
lda = LDA(n_components=2)
X_lda = lda.fit_transform(X_scaled, y)
df_lda = pd.DataFrame(X_lda, columns=['LD1', 'LD2'])

# Display reduced feature heads
print("\n--- PCA Reduced Features (Top 5) ---")
print(df_pca.head())
print("\n--- LDA Reduced Features (Top 5) ---")
print(df_lda.head())


# 5. Compare Classification Accuracy
def evaluate_model(X_data, y_data, name):
    X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.3, random_state=42)
    clf = LogisticRegression()
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    print(f"Accuracy with {name}: {acc:.4f}")
    return acc

print("\n--- Accuracy Comparison ---")
acc_orig = evaluate_model(X_scaled, y, "Original Features (4)")
acc_pca = evaluate_model(X_pca, y, "PCA Features (2)")
acc_lda = evaluate_model(X_lda, y, "LDA Features (2)")


# 6. Plot the 2D Projections
plt.figure(figsize=(14, 6))

# PCA Plot
plt.subplot(1, 2, 1)
for i, target_name in enumerate(le.classes_):
    plt.scatter(X_pca[y == i, 0], X_pca[y == i, 1], label=target_name, alpha=0.7)
plt.title('PCA: Unsupervised Projection')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()

# LDA Plot
plt.subplot(1, 2, 2)
for i, target_name in enumerate(le.classes_):
    plt.scatter(X_lda[y == i, 0], X_lda[y == i, 1], label=target_name, alpha=0.7)
plt.title('LDA: Supervised Projection')
plt.xlabel('Linear Discriminant 1')
plt.ylabel('Linear Discriminant 2')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
#EXP-5&6
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    df = pd.read_csv('HousingData.csv')
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: HousingData.csv not found. Please upload the file to your Colab session.")

df = df.fillna(df.median())

print("\n--- Correlation Heatmap ---")
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, cmap='RdYlGn', fmt='.2f')
plt.title("Feature Correlation Heatmap")
plt.show()

target = 'MEDV'
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

results = []
trained_models = {}

print("\n--- Model Evaluation ---")
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({"Model": name, "MAE": mae, "MSE": mse, "R2 Score": r2})
    trained_models[name] = model
    print(f"{name:20} | R2 Score: {r2:.4f}")

results_df = pd.DataFrame(results).sort_values(by="R2 Score", ascending=False)
print("\nFinal Results:")
display(results_df)

best_model_name = "Gradient Boosting"
importances = trained_models[best_model_name].feature_importances_
feat_importances = pd.Series(importances, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
feat_importances.plot(kind='barh', color='teal')
plt.title(f"Most Influential Features Affecting Property Prices ({best_model_name})")
plt.xlabel("Importance Score")
plt.show()

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.barplot(x="R2 Score", y="Model", data=results_df, palette="viridis")
plt.title("R² Score Comparison (Higher is Better)")
plt.xlim(0, 1)

plt.subplot(1, 2, 2)
sns.barplot(x="MAE", y="Model", data=results_df, palette="magma")
plt.title("MAE Comparison (Lower is Better)")

plt.tight_layout()
plt.show()

best_model = trained_models[best_model_name]
y_pred_best = best_model.predict(X_test_scaled)
residuals = y_test - y_pred_best

plt.figure(figsize=(10, 5))
sns.histplot(residuals, kde=True, color="purple")
plt.axvline(x=0, color='red', linestyle='--')
plt.title(f"Residuals Distribution ({best_model_name})")
plt.xlabel("Error (Actual - Predicted)")
plt.show()

def predict_house_price(input_data):
    """
    Function to take raw input, scale it, and predict price.
    Input should be a list/array in the order of:
    [CRIM, ZN, INDUS, CHAS, NOX, RM, AGE, DIS, RAD, TAX, PTRATIO, B, LSTAT]
    """
    sample_df = pd.DataFrame([input_data], columns=X.columns)

    sample_scaled = scaler.transform(sample_df)

    prediction = best_model.predict(sample_scaled)
    return prediction[0]

sample_house = X_test.iloc[0].values
actual_price = y_test.iloc[0]
predicted_price = predict_house_price(sample_house)

print("\n" + "="*40)
print(f"SAMPLE PREDICTION (Using {best_model_name})")
print("="*40)
print(f"Actual Price:    {actual_price:.2f}")
print(f"Predicted Price: {predicted_price:.2f}")
print(f"Difference:      {abs(actual_price - predicted_price):.2f}")
print("="*40)

best_row = results_df.iloc[0]
print(f"\nCONCLUSION:")
print(f"The best model for Pune Housing is the {best_row['Model']}.")
print(f"It captures {best_row['R2 Score']*100:.2f}% of the price variation.")
print(f"On average, its predictions are off by only {best_row['MAE']:.2f} units.")

Dataset loaded successfully.

--- Correlation Heatmap ---

--- Model Evaluation ---
Linear Regression    | R2 Score: 0.6591
Decision Tree        | R2 Score: 0.6810
Random Forest        | R2 Score: 0.8875
Gradient Boosting    | R2 Score: 0.9002

Final Results:
Model	MAE	MSE	R2 Score
3	Gradient Boosting	1.922403	7.320872	0.900171
2	Random Forest	2.066794	8.250247	0.887497
1	Decision Tree	2.895098	23.393039	0.681006
0	Linear Regression	3.148737	24.999385	0.659101

/tmp/ipython-input-1306/3312981528.py:92: FutureWarning:

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x="R2 Score", y="Model", data=results_df, palette="viridis")
/tmp/ipython-input-1306/3312981528.py:98: FutureWarning:

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x="MAE", y="Model", data=results_df, palette="magma")


========================================
SAMPLE PREDICTION (Using Gradient Boosting)
========================================
Actual Price:    23.60
Predicted Price: 23.23
Difference:      0.37
========================================

CONCLUSION:
The best model for Pune Housing is the Gradient Boosting.
It captures 90.02% of the price variation.
On average, its predictions are off by only 1.92 units.

In [ ]:
#EXP-7
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, precision_score, recall_score, f1_score

df = pd.read_csv('Bank_Loan_Approval_Dataset.csv')

le_emp = LabelEncoder()
df['Employment_Type'] = le_emp.fit_transform(df['Employment_Type'])

le_prop = LabelEncoder()
df['Property_Area'] = le_prop.fit_transform(df['Property_Area'])

X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap of Bank Loan Features')
plt.tight_layout()
plt.show()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

kernels = ['linear', 'rbf', 'poly']
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [1, 0.1, 0.01, 0.001],
}

results = []

print("Starting SVM Training and Tuning...")

for kernel in kernels:
    print(f"\n--- Tuning {kernel.upper()} Kernel ---")

    grid = GridSearchCV(SVC(kernel=kernel), param_grid, refit=True, verbose=0, cv=5)
    grid.fit(X_train, y_train)

    y_pred = grid.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    results.append({
        'Kernel': kernel.capitalize(),
        'Best C': grid.best_params_['C'],
        'Best Gamma': grid.best_params_['gamma'],
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-score': f1
    })

    print(f"Best Parameters: {grid.best_params_}")
    print(f"Accuracy: {acc:.2%}")
    print(f"Confusion Matrix:\n{cm}")
    print(classification_report(y_test, y_pred, zero_division=0))

results_df = pd.DataFrame(results)
print("\n--- Kernel Performance Comparison Table ---")
print(results_df)

results_df.set_index('Kernel')[['Accuracy', 'Precision', 'Recall', 'F1-score']].plot(kind='bar', figsize=(10, 6))
plt.title('SVM Kernel Performance Comparison')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


Starting SVM Training and Tuning...

--- Tuning LINEAR Kernel ---
Best Parameters: {'C': 10, 'gamma': 1}
Accuracy: 80.00%
Confusion Matrix:
[[22  4]
 [ 2  2]]
              precision    recall  f1-score   support

           0       0.92      0.85      0.88        26
           1       0.33      0.50      0.40         4

    accuracy                           0.80        30
   macro avg       0.62      0.67      0.64        30
weighted avg       0.84      0.80      0.82        30


--- Tuning RBF Kernel ---
Best Parameters: {'C': 100, 'gamma': 0.01}
Accuracy: 86.67%
Confusion Matrix:
[[24  2]
 [ 2  2]]
              precision    recall  f1-score   support

           0       0.92      0.92      0.92        26
           1       0.50      0.50      0.50         4

    accuracy                           0.87        30
   macro avg       0.71      0.71      0.71        30
weighted avg       0.87      0.87      0.87        30


--- Tuning POLY Kernel ---
Best Parameters: {'C': 1, 'gamma': 0.1}
Accuracy: 86.67%
Confusion Matrix:
[[26  0]
 [ 4  0]]
              precision    recall  f1-score   support

           0       0.87      1.00      0.93        26
           1       0.00      0.00      0.00         4

    accuracy                           0.87        30
   macro avg       0.43      0.50      0.46        30
weighted avg       0.75      0.87      0.80        30


--- Kernel Performance Comparison Table ---
   Kernel  Best C  Best Gamma  Accuracy  Precision  Recall  F1-score
0  Linear      10        1.00  0.800000   0.333333     0.5       0.4
1     Rbf     100        0.01  0.866667   0.500000     0.5       0.5
2    Poly       1        0.10  0.866667   0.000000     0.0       0.0


In [ ]:
#EXP-8
# 1. Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import zipfile
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 2. Upload File Section
print("Please upload 'bank-additional.zip' below:")
uploaded = files.upload()

# 3. Extract the ZIP file
zip_filename = 'bank-additional.zip'
if os.path.exists(zip_filename):
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("File unzipped successfully.")
else:
    print("Error: Zip file not found. Please ensure the filename is exactly 'bank-additional.zip'.")

# 4. Load the dataset
# Based on the structure: bank-additional.zip/bank-additional/bank-additional-full.csv
file_path = 'bank-additional/bank-additional-full.csv'

if os.path.exists(file_path):
    # Dataset uses ';' as a separator [cite: 10]
    df = pd.read_csv(file_path, sep=';')
    print("Dataset loaded successfully!")
    print(f"Number of Instances: {len(df)}") # [cite: 14]
else:
    print(f"Error: Could not find {file_path}. Check if the folder structure matches.")

# --- EDA & Modeling ---
if 'df' in locals():
    # 5. Preprocessing & Feature Selection
    # Dropping 'duration' for a realistic predictive model [cite: 22]
    df_model = df.drop(columns=['duration'])

    # Convert target 'y' to binary [cite: 13, 25]
    df_model['y'] = df_model['y'].map({'yes': 1, 'no': 0})

    # One-Hot Encode categorical variables [cite: 16, 17, 18]
    df_final = pd.get_dummies(df_model)

    # 6. Model Building
    X = df_final.drop('y', axis=1)
    y = df_final['y']

    # 80/20 Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Random Forest with class weighting to handle imbalance
    model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    model.fit(X_train, y_train)

    # 7. Results
    y_pred = model.predict(X_test)
    print("\n--- Model Performance ---")
    print(classification_report(y_test, y_pred))

    # Visualize Top Features (Social/Economic context) [cite: 7, 9]
    feat_importances = pd.Series(model.feature_importances_, index=X.columns)
    plt.figure(figsize=(10,6))
    feat_importances.nlargest(10).plot(kind='barh', color='skyblue')
    plt.title("Top 10 Predictors for Term Deposit Subscription")
    plt.xlabel("Relative Importance")
    plt.show()

Please upload 'bank-additional.zip' below:
Upload widget is only available when the cell has been executed in the current browser session. Please rerun this cell to enable.
Saving bank-additional.zip to bank-additional.zip
File unzipped successfully.
Dataset loaded successfully!
Number of Instances: 41188

--- Model Performance ---
              precision    recall  f1-score   support

           0       0.91      0.97      0.94      7310
           1       0.57      0.28      0.38       928

    accuracy                           0.90      8238
   macro avg       0.74      0.63      0.66      8238
weighted avg       0.88      0.90      0.88      8238



# 1. Create a sample customer profile (adjust these values to test different scenarios)
# Note: We must match the exact columns created by pd.get_dummies in the previous step
sample_customer = X_test.iloc[[0]].copy()

# 2. Use the model to predict (0 = No, 1 = Yes)
prediction = model.predict(sample_customer)
probability = model.predict_proba(sample_customer)

# 3. Output the result
result = "Yes" if prediction[0] == 1 else "No"
confidence = probability[0][prediction[0]] * 100

print(f"--- Prediction Result ---")
print(f"Will the customer subscribe? {result}")
print(f"Model Confidence: {confidence:.2f}%")

--- Prediction Result ---
Will the customer subscribe? No
Model Confidence: 90.00%

# 1. Generate predictions for all customers in the test set
test_predictions = model.predict(X_test)
test_probabilities = model.predict_proba(X_test)[:, 1] # Probability of 'Yes'

# 2. Create a summary DataFrame
results_df = pd.DataFrame({
    'Actual_Outcome': y_test.values,
    'Predicted_Outcome': test_predictions,
    'Subscription_Probability': test_probabilities
})

# 3. Sort by highest probability to find "Hot Leads"
hot_leads = results_df.sort_values(by='Subscription_Probability', ascending=False)

print("Top 5 Customers Most Likely to Subscribe:")
print(hot_leads.head())

Top 5 Customers Most Likely to Subscribe:
      Actual_Outcome  Predicted_Outcome  Subscription_Probability
1611               1                  1                      1.00
7956               1                  1                      0.98
4500               1                  1                      0.97
3538               1                  1                      0.96
3272               1                  1                      0.96

In [ ]:
#EXP-8
# 1. Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import zipfile
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 2. Upload File Section
print("Please upload 'bank-additional.zip' below:")
uploaded = files.upload()

# 3. Extract the ZIP file
zip_filename = 'bank-additional.zip'
if os.path.exists(zip_filename):
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("File unzipped successfully.")
else:
    print("Error: Zip file not found. Please ensure the filename is exactly 'bank-additional.zip'.")

# 4. Load the dataset
# Based on the structure: bank-additional.zip/bank-additional/bank-additional-full.csv
file_path = 'bank-additional/bank-additional-full.csv'

if os.path.exists(file_path):
    # Dataset uses ';' as a separator [cite: 10]
    df = pd.read_csv(file_path, sep=';')
    print("Dataset loaded successfully!")
    print(f"Number of Instances: {len(df)}") # [cite: 14]
else:
    print(f"Error: Could not find {file_path}. Check if the folder structure matches.")

# --- EDA & Modeling ---
if 'df' in locals():
    # 5. Preprocessing & Feature Selection
    # Dropping 'duration' for a realistic predictive model [cite: 22]
    df_model = df.drop(columns=['duration'])

    # Convert target 'y' to binary [cite: 13, 25]
    df_model['y'] = df_model['y'].map({'yes': 1, 'no': 0})

    # One-Hot Encode categorical variables [cite: 16, 17, 18]
    df_final = pd.get_dummies(df_model)

    # 6. Model Building
    X = df_final.drop('y', axis=1)
    y = df_final['y']

    # 80/20 Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Random Forest with class weighting to handle imbalance
    model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    model.fit(X_train, y_train)

    # 7. Results
    y_pred = model.predict(X_test)
    print("\n--- Model Performance ---")
    print(classification_report(y_test, y_pred))

    # Visualize Top Features (Social/Economic context) [cite: 7, 9]
    feat_importances = pd.Series(model.feature_importances_, index=X.columns)
    plt.figure(figsize=(10,6))
    feat_importances.nlargest(10).plot(kind='barh', color='skyblue')
    plt.title("Top 10 Predictors for Term Deposit Subscription")
    plt.xlabel("Relative Importance")
    plt.show()

Please upload 'bank-additional.zip' below:
Upload widget is only available when the cell has been executed in the current browser session. Please rerun this cell to enable.
Saving bank-additional.zip to bank-additional.zip
File unzipped successfully.
Dataset loaded successfully!
Number of Instances: 41188

--- Model Performance ---
              precision    recall  f1-score   support

           0       0.91      0.97      0.94      7310
           1       0.57      0.28      0.38       928

    accuracy                           0.90      8238
   macro avg       0.74      0.63      0.66      8238
weighted avg       0.88      0.90      0.88      8238



# 1. Create a sample customer profile (adjust these values to test different scenarios)
# Note: We must match the exact columns created by pd.get_dummies in the previous step
sample_customer = X_test.iloc[[0]].copy()

# 2. Use the model to predict (0 = No, 1 = Yes)
prediction = model.predict(sample_customer)
probability = model.predict_proba(sample_customer)

# 3. Output the result
result = "Yes" if prediction[0] == 1 else "No"
confidence = probability[0][prediction[0]] * 100

print(f"--- Prediction Result ---")
print(f"Will the customer subscribe? {result}")
print(f"Model Confidence: {confidence:.2f}%")

--- Prediction Result ---
Will the customer subscribe? No
Model Confidence: 90.00%

# 1. Generate predictions for all customers in the test set
test_predictions = model.predict(X_test)
test_probabilities = model.predict_proba(X_test)[:, 1] # Probability of 'Yes'

# 2. Create a summary DataFrame
results_df = pd.DataFrame({
    'Actual_Outcome': y_test.values,
    'Predicted_Outcome': test_predictions,
    'Subscription_Probability': test_probabilities
})

# 3. Sort by highest probability to find "Hot Leads"
hot_leads = results_df.sort_values(by='Subscription_Probability', ascending=False)

print("Top 5 Customers Most Likely to Subscribe:")
print(hot_leads.head())

Top 5 Customers Most Likely to Subscribe:
      Actual_Outcome  Predicted_Outcome  Subscription_Probability
1611               1                  1                      1.00
7956               1                  1                      0.98
4500               1                  1                      0.97
3538               1                  1                      0.96
3272               1                  1                      0.96

In [ ]:
#EXP-9

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import matplotlib.pyplot as plt

# 1. Recreate the dataset from the image
data = {
    'Day': ['D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14'],
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Overcast', 'Rain'],
    'Temp': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
    'Play Tennis': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
}

df = pd.DataFrame(data)
print("--- Original Dataset ---")
print(df.to_string(index=False))
print("\n" + "="*50 + "\n")

# 2. Preprocessing: Convert categorical text data to numerical codes
# Decision Trees in sklearn require numerical input
df_encoded = df.copy()
for col in ['Outlook', 'Temp', 'Humidity', 'Wind']:
    df_encoded[col] = df_encoded[col].astype('category').cat.codes

# Target variable mapping: 'No' -> 0, 'Yes' -> 1
df_encoded['Play Tennis'] = df_encoded['Play Tennis'].map({'No': 0, 'Yes': 1})

# Define features (X) and target (y)
X = df_encoded[['Outlook', 'Temp', 'Humidity', 'Wind']]
y = df_encoded['Play Tennis']

# 3. Train/Test Split
# Using a small test size (30%) since we only have 14 rows.
# random_state ensures you get the exact same results every time you run it.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 4. Build the Decision Tree Model
# Using 'entropy' (Information Gain) as it's the standard for this classic dataset
clf = DecisionTreeClassifier(criterion='entropy', random_state=42)
clf.fit(X_train, y_train)

# 5. Predict on the test set
y_pred = clf.predict(X_test)

# 6. Evaluate Performance
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("--- Model Performance ---")
print(f"Accuracy:  {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F1-Score:  {f1:.2f}")
print("\n--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['No (0)', 'Yes (1)']))
print("="*50 + "\n")

# 7. Interpret the Results (Visualizing the tree)
plt.figure(figsize=(12, 8))
plot_tree(clf,
          feature_names=['Outlook', 'Temp', 'Humidity', 'Wind'],
          class_names=['No', 'Yes'],
          filled=True,
          rounded=True)
plt.title("Decision Tree for 'Play Tennis'")
plt.show()

--- Original Dataset ---
Day  Outlook Temp Humidity   Wind Play Tennis
 D1    Sunny  Hot     High   Weak          No
 D2    Sunny  Hot     High Strong          No
 D3 Overcast  Hot     High   Weak         Yes
 D4     Rain Mild     High   Weak         Yes
 D5     Rain Cool   Normal   Weak         Yes
 D6     Rain Cool   Normal Strong          No
 D7 Overcast Cool   Normal Strong         Yes
 D8    Sunny Mild     High   Weak          No
 D9    Sunny Cool   Normal   Weak         Yes
D10     Rain Mild   Normal   Weak         Yes
D11    Sunny Mild   Normal Strong         Yes
D12 Overcast Mild     High Strong         Yes
D13 Overcast  Hot   Normal   Weak         Yes
D14     Rain Mild     High Strong          No

==================================================

--- Model Performance ---
Accuracy:  0.60
Precision: 0.67
Recall:    0.67
F1-Score:  0.67

--- Detailed Classification Report ---
              precision    recall  f1-score   support

      No (0)       0.50      0.50      0.50         2
     Yes (1)       0.67      0.67      0.67         3

    accuracy                           0.60         5
   macro avg       0.58      0.58      0.58         5
weighted avg       0.60      0.60      0.60         5

==================================================



In [ ]:
#EXP-10
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules
import warnings

# 1. Setup & Clean Output
# This hides the 'datetime' deprecation warnings you were seeing
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# 2. Load the Dataset
# Using a direct raw link to the Online Retail CSV on GitHub
url = "https://raw.githubusercontent.com/StephanT324/Market_Basket_Analysis/master/OnlineRetail.csv"

print("Fetching data from verified GitHub mirror...")
try:
    # Reading with 'latin1' encoding to handle special characters in product names
    df = pd.read_csv(url, encoding='latin1')
    print("✓ Data loaded successfully!")

    # 3. Data Cleaning
    print("Processing and cleaning...")
    df['Description'] = df['Description'].str.strip()
    df.dropna(axis=0, subset=['InvoiceNo'], inplace=True)
    df['InvoiceNo'] = df['InvoiceNo'].astype('str')

    # Remove credit transactions (Invoice numbers starting with 'C')
    df = df[~df['InvoiceNo'].str.contains('C')]

    # 4. Create the 'Basket' (Using France as a sample for performance)
    basket = (df[df['Country'] =="France"]
              .groupby(['InvoiceNo', 'Description'])['Quantity']
              .sum().unstack().reset_index().fillna(0)
              .set_index('InvoiceNo'))

    # 5. One-Hot Encoding
    # The algorithm requires 0 or 1 (Present/Absent)
    def encode_units(x):
        return 1 if x >= 1 else 0

    basket_sets = basket.applymap(encode_units)

    # Drop 'POSTAGE' as it's a shipping fee, not a product
    if 'POSTAGE' in basket_sets.columns:
        basket_sets.drop('POSTAGE', inplace=True, axis=1)

    # 6. Apply Apriori Algorithm
    # min_support=0.07 means the item appears in at least 7% of orders
    frequent_itemsets = apriori(basket_sets, min_support=0.07, use_colnames=True)

    # 7. Generate Association Rules
    # We use 'lift' to find the strongest relationships
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
    rules = rules.sort_values(['lift', 'confidence'], ascending=[False, False])

    # 8. Output Top 10 Results
    print("\n--- Top 10 Product Associations found in France ---")
    print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

    # 9. Visualization
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=rules['support'], y=rules['confidence'],
                    size=rules['lift'], hue=rules['lift'], palette='plasma')
    plt.title('Market Basket Analysis: Support vs. Confidence')
    plt.xlabel('Support (How common the pair is)')
    plt.ylabel('Confidence (Probability of buying B if A is bought)')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

except Exception as e:
    print(f"\n[ERROR]: {e}")
    print("If you still see a 404, please try running: !wget {url} to check connectivity.")

In [ ]:
#EXP-11
1. Data Preparation First, we load the data and split it into training and testing sets.


import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Creating the dataset from the image
data = {
    'Age': [22, 25, 28, 30, 35, 40, 45, 50, 23, 27, 33, 38, 42, 48, 29, 31, 36, 44, 26, 39],
    'Income': [25000, 30000, 35000, 40000, 50000, 60000, 65000, 70000, 27000, 32000,
               48000, 55000, 62000, 68000, 36000, 42000, 52000, 64000, 31000, 58000],
    'Browsing_Time': [5, 10, 15, 20, 25, 30, 10, 12, 8, 18, 22, 28, 16, 14, 19, 21, 24, 11, 13, 27],
    'Pages_Viewed': [3, 5, 7, 10, 12, 15, 6, 8, 4, 9, 11, 14, 7, 6, 9, 10, 12, 5, 6, 13],
    'Purchase': [0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1]
}

df = pd.DataFrame(data)
X = df.drop('Purchase', axis=1)
y = df['Purchase']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

2. Model Implementation

● Decision Tree using Bagging Bagging (Bootstrap Aggregating) reduces variance by training multiple versions of the model on different subsets of the data.


bagging_model = BaggingClassifier(estimator=DecisionTreeClassifier(),
                                  n_estimators=10, random_state=42)
bagging_model.fit(X_train, y_train)
bag_pred = bagging_model.predict(X_test)
print(f"Bagging Accuracy: {accuracy_score(y_test, bag_pred)}")

● AdaBoost Classifier AdaBoost (Adaptive Boosting) focuses on instances that the previous models misclassified, assigning them higher weights to improve accuracy.


adaboost_model = AdaBoostClassifier(n_estimators=50, random_state=42)
adaboost_model.fit(X_train, y_train)
ada_pred = adaboost_model.predict(X_test)
print(f"AdaBoost Accuracy: {accuracy_score(y_test, ada_pred)}")

● Stacking Model Stacking uses a "Meta-Learner" (usually Logistic Regression) to combine the predictions of multiple base models (Decision Tree and SVM).


# Define base learners
base_learners = [
    ('dt', DecisionTreeClassifier()),
    ('svc', SVC(probability=True)),
    ('lr', LogisticRegression())
]

# Define Stacking model with Logistic Regression as the final estimator
stacking_model = StackingClassifier(estimators=base_learners,
                                    final_estimator=LogisticRegression())
stacking_model.fit(X_train, y_train)
stack_pred = stacking_model.predict(X_test)
print(f"Stacking Accuracy: {accuracy_score(y_test, stack_pred)}")
